# Contracts over canonical interval formulas

`Contract` is an application-level wrapper. Its logical content is an implication; it does not introduce a second interval-logic AST or evaluation semantics.

In [ ]:
from dataclasses import dataclass

import portion as P
import interval_logic as il

@dataclass(frozen=True)
class Contract:
    assumption: il.Formula | bool | str
    guarantee: il.Formula | bool | str

    def __post_init__(self):
        object.__setattr__(self, "assumption", il.as_formula(self.assumption).simplify())
        object.__setattr__(self, "guarantee", il.as_formula(self.guarantee).simplify())

    @property
    def formula(self) -> il.Formula:
        return self.assumption.implies(self.guarantee)

    def replace(self, *functions) -> "Contract":
        return Contract(
            self.assumption.replace(*functions),
            self.guarantee.replace(*functions),
        )

## Charger requests and commitments

A flexible request asks for a start in an absolute interval, then re-anchors a half-open dwell condition at that start. A concrete commitment uses an absolute condition over its reserved interval. Durations remain physical time, never event counts.

In [ ]:
@dataclass(frozen=True)
class Request:
    job: int
    duration: float
    start_interval: P.Interval

    def to_contract(self) -> Contract:
        assigned = il.Signal("job").equals(self.job)
        dwell = assigned.always(self.duration)
        return Contract(True, dwell.eventually(self.start_interval))

@dataclass(frozen=True)
class Commitment:
    job: int
    interval: P.Interval

    def to_contract(self) -> Contract:
        assigned = il.Signal("job").equals(self.job)
        return Contract(True, assigned.always(self.interval))

request = Request(2, 1.0, P.closedopen(2, 4)).to_contract()
commitment = Commitment(1, P.closedopen(1, 3)).to_contract()
print(request.formula)
print(commitment.formula)

## Checking a complete schedule

A known schedule is a `Trace`. This notebook checks only complete represented schedules.

In [ ]:
trace = il.Trace(
    P.closed(0, 6),
    signals={
        "job": il.NaturalSignal({
            0: P.closedopen(0, 1) | P.closedopen(3, 4),
            1: P.closedopen(1, 3),
            2: P.closed(4, 6),
        })
    },
)

print("request:", il.satisfies(request.guarantee, trace, 0))
print("commitment:", il.satisfies(commitment.guarantee, trace, 0))